 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : Martin Law                         <br>
Student Number : L00203482                          <br>
Due Date       : Tuesday 12th May 2026              <br>
Assignment     : CA2                                <br>
Module         : AI for Vision and NLP              <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level

This application will analyse scanned or photographed document images using a combined computer vision, OCR, NLP, and reporting pipeline.

The planned high level process is:

1. load scanned or photographed document images
2. preprocess images using OpenCV
3. extract text using Tesseract OCR
4. compare OCR results across different preprocessed image versions
5. clean and process extracted text
6. apply tokenisation, stopword removal, stemming, and lemmatisation
7. extract NLP features such as tokens, stems, lemmas, keywords, and named entities
8. detect visual document features such as text blocks, tables, signatures, diagrams, figures, and form-like regions
9. combine the text and visual findings
10. export structured document summaries, annotated images, CSV files, and JSON reports

At this development stage, the notebook now adds text analysis and section categorisation. Visual feature detection and image segmentation are left for later phases

In [ ]:
# high-level pipeline placeholder for the first development stage

pipeline_steps = [
    "load scanned or photographed document images",
    "preprocess images using OpenCV",
    "extract text using Tesseract OCR",
    "compare OCR confidence across preprocessing methods",
    "select the best OCR result per doc"
    "clean extracted OCR text",
    "tokenise cleaned text",
    "remove stopwords",
    "apply stemming",
    "apply lemmatisation",
    "extract TF-IDF keywords",
    "extract simple key phrases",
    "extract regex-based named entities and document references",
    "save NLP preprocessing outputs"
]

for step_number, step_description in enumerate(pipeline_steps, start=1):
    print(f"{step_number}. {step_description}")

# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [ ]:
# pip installs
# run this cell only if the required packages are not already instlal ed in the selected environment

# %pip install opencv-python pytesseract pandas numpy matplotlib scikit-learn nltk jupyter ipykernel

## Imports

In [ ]:
# imports

from pathlib import Path
from collections import Counter
import os # had to add this due to tesseract issues on windows
import platform
import json
import shutil
import re

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytesseract
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

# Support Functions

In [ ]:
# support functions and project folders

PROJECT_DIR = Path.cwd()
INPUT_DIR = PROJECT_DIR / "data" / "input_documents"
OUTPUT_DIR = PROJECT_DIR / "outputs"

PREPROCESSED_DIR = OUTPUT_DIR / "preprocessed_images"
OCR_DIR = OUTPUT_DIR / "ocr_results"
OCR_TEXT_DIR = OUTPUT_DIR / "ocr_text"
NLP_DIR = OUTPUT_DIR / "nlp_results"
CLEANED_TEXT_DIR = OUTPUT_DIR / "cleaned_text"
TEXT_FEATURE_DIR = OUTPUT_DIR / "text_features"
SECTION_ANALYSIS_DIR = OUTPUT_DIR / "section_analysis"
VISUAL_FEATURE_DIR = OUTPUT_DIR / "visual_features"
SEGMENTATION_DIR = OUTPUT_DIR / "segmentation"
ANNOTATED_DIR = OUTPUT_DIR / "annotated_images"
JSON_DIR = OUTPUT_DIR / "json_reports"
CSV_DIR = OUTPUT_DIR / "csv_reports"

def create_project_folders():
    """ create the folder structure used by this notebook"""
    folders = [
        INPUT_DIR,
        OUTPUT_DIR,
        PREPROCESSED_DIR,
        OCR_DIR,
        OCR_TEXT_DIR,
        NLP_DIR,
        CLEANED_TEXT_DIR,
        TEXT_FEATURE_DIR,
        SECTION_ANALYSIS_DIR,
        VISUAL_FEATURE_DIR,
        SEGMENTATION_DIR,
        ANNOTATED_DIR,
        JSON_DIR,
        CSV_DIR
    ]

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)

    return folders

def list_input_documents():
    """return supported image files from the input document folder"""
    supported_extensions = [
        "*.jpg",
        "*.jpeg",
        "*.png",
        "*.bmp",
        "*.tif",
        "*.tiff"    
    ]

    image_paths = []

    for extension in supported_extensions:
        image_paths.extend(INPUT_DIR.glob(extension))

    return sorted(image_paths)

created_folders = create_project_folders()
input_documents = list_input_documents()

print("Project directory:", PROJECT_DIR)
print("Operating system:", platform.system())
print("Input directory:", INPUT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Preprocessed image directory:", PREPROCESSED_DIR)
print("OCR results directory:", OCR_DIR)
print("NLP results directory:", NLP_DIR)
print("Text feature directory:", TEXT_FEATURE_DIR)
print("Section analysis directory:", SECTION_ANALYSIS_DIR)
print("Visual feature directory:", VISUAL_FEATURE_DIR)
print("Segmentation directory:", SEGMENTATION_DIR)
print("Number of input document images found:", len(input_documents))

for image_path in input_documents:
    print("-", image_path.name)

# NLP

## Stage 5, Stage 6, and Stage 7: OCR Text Preprocessing, Feature Recognition, and Text Analysis

This section processes the text extracted by Tesseract OCR.

Stage 5 performs the required text preprocessing:

1. clean the raw OCR text
2. convert text to lowercase
3. tokenise the text
4. remove stopwords
5. apply stemming
6. apply lemmatisation
7. save the cleaned text and preprocessing outputs

Stage 6 adds feature recognition:

1. TF-IDF keyword extraction
2. simple key phrase extraction
3. regex-based named entity extraction
4. references to tables, figures, diagrams, and signatures

Stage 7 adds text analysis:
1. reconstruct OCR lines from word-level OCR output
2. classify OCR lines into document sections
3. extract simple table-like rows
4. identify text references to visual document elements

Visual feature detection and image segmentation are deliberately left for later phases

In [ ]:
# NLP preprocessing functions

# NLTK data is downloaded only if it is missing from the current environment
def ensure_nltk_resource(resource_path, download_name):
    """download an NLTK resource only when it is not already available"""
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(download_name, quiet=True)


ensure_nltk_resource("corpora/stopwords", "stopwords")
ensure_nltk_resource("corpora/wordnet", "wordnet")
ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")

STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()


def normalise_text_spacing(text):
    """normalise spacing in OCR text without removing meaningful characters"""
    text = str(text)
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenise_ocr_text(text):
    """tokenise OCR text using a regex that keeps words, numbers, and euro amounts"""
    text = text.lower()

    tokens = re.findall(
        r"€\s*\d+(?:[.,]\d+)?|[a-zA-Z]+(?:'[a-z]+)?|\d+(?:[.,]\d+)?",
        text
    )

    tokens = [token.replace(" ", "") for token in tokens]

    return tokens


def remove_stopwords(tokens):
    """remove common English stopwords and very short alphabetic tokens"""
    filtered_tokens = []

    for token in tokens:
        is_short_word = token.isalpha() and len(token) <= 1

        if token not in STOP_WORDS and not is_short_word:
            filtered_tokens.append(token)

    return filtered_tokens


def stem_tokens(tokens):
    """apply Porter stemming to alphabetic tokens"""
    stemmed_tokens = []

    for token in tokens:
        if token.isalpha():
            stemmed_tokens.append(STEMMER.stem(token))
        else:
            stemmed_tokens.append(token)

    return stemmed_tokens


def lemmatise_tokens(tokens):
    """apply WordNet lemmatisation to alphabetic tokens"""
    lemmatised_tokens = []

    for token in tokens:
        if token.isalpha():
            lemmatised_tokens.append(LEMMATIZER.lemmatize(token))
        else:
            lemmatised_tokens.append(token)

    return lemmatised_tokens


def preprocess_ocr_text(raw_text):
    """run the full Phase 5 preprocessing sequence on OCR text"""
    normalised_text = normalise_text_spacing(raw_text)
    lowercase_text = normalised_text.lower()
    tokens = tokenise_ocr_text(lowercase_text)
    tokens_without_stopwords = remove_stopwords(tokens)
    stems = stem_tokens(tokens_without_stopwords)
    lemmas = lemmatise_tokens(tokens_without_stopwords)
    cleaned_text = " ".join(lemmas)

    return {
        "raw_text": raw_text,
        "normalised_text": normalised_text,
        "lowercase_text": lowercase_text,
        "tokens": tokens,
        "tokens_without_stopwords": tokens_without_stopwords,
        "stems": stems,
        "lemmas": lemmas,
        "cleaned_text": cleaned_text,
        "raw_character_count": len(str(raw_text)),
        "normalised_character_count": len(normalised_text),
        "token_count": len(tokens),
        "tokens_without_stopwords_count": len(tokens_without_stopwords),
        "stem_count": len(stems),
        "lemma_count": len(lemmas)
    }

In [ ]:
# NLP development status for this commit

nlp_development_status = {
    "ocr_text_available": "implemented",
    "ocr_text_cleaning": "implemented",
    "tokenisation": "implemented",
    "stopword_removal": "implemented",
    "stemming": "implemented",
    "lemmatisation": "implemented",
    "tfidf_keyword_extraction": "implemented",
    "named_entity_extraction": "implemented",
    "figure_table_reference_detection": "implemented"
}

nlp_development_status

# Vision

## Stage 3: Image Preprocessing and Enhancement

This section prepares scanned or photographed document images for later OCR and feature detection.

The preprocessing steps are:

1. load the document image
2. convert it to grayscale
3. reduce noise using median blur
4. enhance contrast using CLAHE
5. apply Otsu thresholding
6. apply adaptive thresholding
7. apply a cleaner adaptive threshold for OCR comparison
8. detect edges using Canny edge detection
9. save the processed images for later stages

The OCR phase below will compare several of these preprocessed outputs and select the one with the best tereract confidence score

In [ ]:
# load and display input document images

input_documents = list_input_documents()

if len(input_documents) == 0:
    raise FileNotFoundError(
        f"No document images were found in {INPUT_DIR}. "
        "Add .jpg, .jpeg, .png, .bmp, .tif, or .tiff files before running this cell"
    )

loaded_documents = []

for image_path in input_documents:
    image = cv2.imread(str(image_path))

    if image is None:
        print(f"Could not load image: {image_path.name}")
        continue

    loaded_documents.append({
        "path": image_path,
        "filename": image_path.name,
        "image": image,
        "height": image.shape[0],
        "width": image.shape[1],
        "channels": image.shape[2] if len(image.shape) == 3 else 1
    })

print("Loaded document images:")

for document in loaded_documents:
    print(
        f"- {document['filename']} "
        f"({document['width']} x {document['height']}, "
        f"{document['channels']} channel(s))"
    )

In [ ]:
# display a sample of the loaded document images

documents_to_display = loaded_documents[:4]

plt.figure(figsize=(14, 10))

for index, document in enumerate(documents_to_display, start=1):
    image_rgb = cv2.cvtColor(document["image"], cv2.COLOR_BGR2RGB)

    plt.subplot(2, 2, index)
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.title(document["filename"])

plt.tight_layout()
plt.show()

In [ ]:
# OpenCV preprocessing functions

def preprocess_document_image(image):
    """Apply basic OpenCV preprocessing steps to a document image."""
    if image is None:
        raise ValueError("Cannot preprocess an empty image.")

    # convert from colour to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # reduce small noise while keeping text edges reasonably sharp
    denoised = cv2.medianBlur(gray, 3)

    # improve local contrast, useful for photographed documents with uneven lighting
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(denoised)

    # global thresholding using Otsu's method
    _, otsu_threshold = cv2.threshold(
        enhanced,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # adaptive thresholding handles uneven lighting better than a single global threshold
    adaptive_threshold = cv2.adaptiveThreshold(
        enhanced,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        15
    )

    # a softer adaptive threshold version for OCR comparison
    adaptive_threshold_clean = cv2.adaptiveThreshold(
        enhanced,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        51,
        11
    )

    # remove small isolated black noise from the thresholded image
    noise_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

    adaptive_threshold_clean = cv2.morphologyEx(
        adaptive_threshold_clean,
        cv2.MORPH_OPEN,
        noise_kernel
    )

    # edge detection is useful later for detecting boxes, tables, and page boundaries
    edges = cv2.Canny(
        enhanced,
        threshold1=50,
        threshold2=150
    )

    return {
        "gray": gray,
        "denoised": denoised,
        "enhanced": enhanced,
        "otsu_threshold": otsu_threshold,
        "adaptive_threshold": adaptive_threshold,
        "adaptive_threshold_clean": adaptive_threshold_clean,
        "edges": edges
    }


In [ ]:
# preprocess all loaded document images and save the results

if "loaded_documents" not in globals():
    raise NameError("loaded_documents does not exist. Run the image loading cell before this cell.")

if len(loaded_documents) == 0:
    raise ValueError("No loaded documents are available for preprocessing.")

preprocessed_documents = []

for document in loaded_documents:
    filename_stem = Path(document["filename"]).stem

    processed_images = preprocess_document_image(document["image"])

    output_paths = {}

    for processing_name, processed_image in processed_images.items():
        output_path = PREPROCESSED_DIR / f"{filename_stem}_{processing_name}.png"
        cv2.imwrite(str(output_path), processed_image)
        output_paths[processing_name] = output_path

    preprocessed_documents.append({
        "filename": document["filename"],
        "original_image": document["image"],
        "processed_images": processed_images,
        "output_paths": output_paths
    })

print("Preprocessed document images saved:")

for document in preprocessed_documents:
    print(f"- {document['filename']}")

    for processing_name, output_path in document["output_paths"].items():
        print(f"  {processing_name}: {output_path.name}")


In [ ]:
# display preprocessing results for the first document

example_document = preprocessed_documents[0]

original_rgb = cv2.cvtColor(example_document["original_image"], cv2.COLOR_BGR2RGB)

display_images = [
    ("Original", original_rgb, None),
    ("Grayscale", example_document["processed_images"]["gray"], "gray"),
    ("Denoised", example_document["processed_images"]["denoised"], "gray"),
    ("Enhanced", example_document["processed_images"]["enhanced"], "gray"),
    ("Otsu Threshold", example_document["processed_images"]["otsu_threshold"], "gray"),
    ("Adaptive Threshold", example_document["processed_images"]["adaptive_threshold"], "gray"),
    ("Clean Adaptive Threshold", example_document["processed_images"]["adaptive_threshold_clean"], "gray"),
    ("Canny Edges", example_document["processed_images"]["edges"], "gray")
]

plt.figure(figsize=(16, 12))

for index, (title, image, colour_map) in enumerate(display_images, start=1):
    plt.subplot(3, 3, index)
    plt.imshow(image, cmap=colour_map)
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

print("Displayed preprocessing results for:", example_document["filename"])


## Stage 4: Tesseract OCR Extraction and Confidence Comparison

This stage extracts text from each document image using Tesseract OCR.

To avoid assuming that one preprocessing method is always best, OCR is tested on several image versions:

1. grayscale
2. denoised
3. contrast-enhanced
4. clean adaptive threshold

The notebook then compares the average Tesseract confidence score and selects the best OCR result for each document.

This stage addresses the OCR accuracy part of the rubric. NLP preprocessing is added in the next phase.

In [ ]:
# Tesseract OCR configuration

OCR_CONFIG = "--oem 3 --psm 6"

def configure_tesseract_for_current_system():
    """configure the Tesseract executable path for Windows and macOS"""
    system_name = platform.system()
    possible_paths = []

    if system_name == "Windows":
        possible_paths = [
            Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe"),
            Path(r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"),
            Path(os.environ.get("LOCALAPPDATA", "")) / "Programs" / "Tesseract-OCR" / "tesseract.exe",
        ]

    elif system_name == "Darwin":
        possible_paths = [
            Path("/opt/homebrew/bin/tesseract"),
            Path("/usr/local/bin/tesseract"),
        ]

    else:
        possible_paths = []

    for possible_path in possible_paths:
        if possible_path.exists():
            pytesseract.pytesseract.tesseract_cmd = str(possible_path)
            break
    else:
        detected_path = shutil.which("tesseract")

        if detected_path:
            pytesseract.pytesseract.tesseract_cmd = detected_path

    try:
        version = pytesseract.get_tesseract_version()
        print("Tesseract detected:", version)
        print("Tesseract path:", pytesseract.pytesseract.tesseract_cmd)

    except Exception as error:
        if system_name == "Windows":
            install_message = (
                "Tesseract could not be found by pytesseract. "
                "On Windows, install it with: winget install --id UB-Mannheim.TesseractOCR -e "
                "or set pytesseract.pytesseract.tesseract_cmd to the full tesseract.exe path."
            )

        elif system_name == "Darwin":
            install_message = (
                "Tesseract could not be found by pytesseract. "
                "On macOS, install it with: brew install tesseract."
            )

        else:
            install_message = (
                "Tesseract could not be found by pytesseract. "
                "Install Tesseract OCR and ensure it is available on PATH."
            )

        raise RuntimeError(install_message) from error


configure_tesseract_for_current_system()

In [ ]:
# OCR helper functions

def normalise_ocr_dataframe(ocr_dataframe):
    """clean the dataframe returned by pytesseract.image_to_data"""
    cleaned_dataframe = ocr_dataframe.copy()

    cleaned_dataframe["text"] = cleaned_dataframe["text"].fillna("").astype(str)
    cleaned_dataframe["text"] = cleaned_dataframe["text"].str.strip()
    cleaned_dataframe["conf"] = pd.to_numeric(cleaned_dataframe["conf"], errors="coerce")

    cleaned_dataframe = cleaned_dataframe[
        (cleaned_dataframe["text"] != "") &
        (cleaned_dataframe["conf"].notna()) &
        (cleaned_dataframe["conf"] >= 0)
    ].copy()

    return cleaned_dataframe


def run_ocr_on_image(image):
    """run Tesseract OCR on one image and return text, word data, and confidence metrics"""
    ocr_dataframe = pytesseract.image_to_data(
        image,
        config=OCR_CONFIG,
        output_type=pytesseract.Output.DATAFRAME
    )

    ocr_dataframe = normalise_ocr_dataframe(ocr_dataframe)

    extracted_text = " ".join(ocr_dataframe["text"].tolist())

    if len(ocr_dataframe) == 0:
        average_confidence = 0.0
        median_confidence = 0.0
    else:
        average_confidence = float(ocr_dataframe["conf"].mean())
        median_confidence = float(ocr_dataframe["conf"].median())

    return {
        "text": extracted_text,
        "ocr_dataframe": ocr_dataframe,
        "average_confidence": average_confidence,
        "median_confidence": median_confidence,
        "word_count": int(len(ocr_dataframe)),
        "character_count": int(len(extracted_text))
    }


def select_ocr_candidate_images(preprocessed_document):
    """select the preprocessed image versions used for OCR comparison"""
    processed_images = preprocessed_document["processed_images"]

    candidate_names = [
        "gray",
        "denoised",
        "enhanced",
        "adaptive_threshold_clean"
    ]

    candidates = {}

    for candidate_name in candidate_names:
        if candidate_name in processed_images:
            candidates[candidate_name] = processed_images[candidate_name]

    return candidates

In [ ]:
# run OCR on each selected preprocessing version and select the best result

if "preprocessed_documents" not in globals():
    raise NameError("preprocessed_documents does not exist. Run the preprocessing cells before this OCR cell.")

ocr_results = []

for document in preprocessed_documents:
    filename = document["filename"]
    filename_stem = Path(filename).stem

    candidate_images = select_ocr_candidate_images(document)
    candidate_results = []

    for candidate_name, candidate_image in candidate_images.items():
        result = run_ocr_on_image(candidate_image)

        ocr_csv_path = OCR_DIR / f"{filename_stem}_{candidate_name}_ocr_words.csv"
        result["ocr_dataframe"].to_csv(ocr_csv_path, index=False)

        candidate_results.append({
            "candidate_name": candidate_name,
            "text": result["text"],
            "average_confidence": result["average_confidence"],
            "median_confidence": result["median_confidence"],
            "word_count": result["word_count"],
            "character_count": result["character_count"],
            "ocr_csv_path": str(ocr_csv_path)
        })

    best_candidate = max(
        candidate_results,
        key=lambda item: (
            item["average_confidence"],
            item["word_count"],
            item["character_count"]
        )
    )

    best_text_path = OCR_TEXT_DIR / f"{filename_stem}_best_ocr_text.txt"

    with open(best_text_path, "w", encoding="utf-8") as text_file:
        text_file.write(best_candidate["text"])

    ocr_results.append({
        "filename": filename,
        "candidate_results": candidate_results,
        "best_candidate_name": best_candidate["candidate_name"],
        "best_average_confidence": best_candidate["average_confidence"],
        "best_median_confidence": best_candidate["median_confidence"],
        "best_word_count": best_candidate["word_count"],
        "best_character_count": best_candidate["character_count"],
        "best_text": best_candidate["text"],
        "best_text_path": str(best_text_path)
    })

print("OCR completed for document images:")

for result in ocr_results:
    print(
        f"- {result['filename']}: "
        f"best={result['best_candidate_name']}, "
        f"avg_conf={result['best_average_confidence']:.2f}, "
        f"words={result['best_word_count']}"
    )

In [ ]:
# display OCR confidence comparison

ocr_summary_rows = []

for result in ocr_results:
    for candidate in result["candidate_results"]:
        ocr_summary_rows.append({
            "document": result["filename"],
            "ocr_source": candidate["candidate_name"],
            "average_confidence": round(candidate["average_confidence"], 2),
            "median_confidence": round(candidate["median_confidence"], 2),
            "word_count": candidate["word_count"],
            "character_count": candidate["character_count"],
            "selected_as_best": candidate["candidate_name"] == result["best_candidate_name"]
        })

ocr_summary_df = pd.DataFrame(ocr_summary_rows)

ocr_summary_csv_path = CSV_DIR / "ocr_confidence_summary.csv"
ocr_summary_df.to_csv(ocr_summary_csv_path, index=False)

ocr_summary_df

In [ ]:
# display best OCR text for the first document

example_ocr_result = ocr_results[0]

print("Document:", example_ocr_result["filename"])
print("Best OCR source:", example_ocr_result["best_candidate_name"])
print("Average confidence:", round(example_ocr_result["best_average_confidence"], 2))
print("Median confidence:", round(example_ocr_result["best_median_confidence"], 2))
print("Word count:", example_ocr_result["best_word_count"])
print()
print("Best OCR text preview:")
print("-" * 80)
print(example_ocr_result["best_text"][:2000])

## Stage 5: Apply NLP Preprocessing to OCR Text

This stage applies the NLP preprocessing functions to the best OCR result selected for each document.

The output separates the raw OCR text from the cleaned and processed text so that the effect of tokenisation, stopword removal, stemming, and lemmatisation can be inspected clearly.


In [ ]:
# apply NLP preprocessing to the best OCR result for each document

if "ocr_results" not in globals():
    raise NameError("ocr_results does not exist. Run the OCR cells before this NLP preprocessing cell.")

if len(ocr_results) == 0:
    raise ValueError("No OCR results are available for NLP preprocessing.")

nlp_results = []

for ocr_result in ocr_results:
    filename = ocr_result["filename"]
    filename_stem = Path(filename).stem
    raw_text = ocr_result["best_text"]

    processed_text = preprocess_ocr_text(raw_text)

    nlp_json_path = NLP_DIR / f"{filename_stem}_nlp_preprocessing.json"
    cleaned_text_path = CLEANED_TEXT_DIR / f"{filename_stem}_cleaned_text.txt"

    nlp_record = {
        "filename": filename,
        "best_ocr_source": ocr_result["best_candidate_name"],
        "best_ocr_average_confidence": ocr_result["best_average_confidence"],
        "best_ocr_median_confidence": ocr_result["best_median_confidence"],
        "preprocessing": processed_text,
        "output_paths": {
            "nlp_json": str(nlp_json_path),
            "cleaned_text": str(cleaned_text_path)
        }
    }

    with open(nlp_json_path, "w", encoding="utf-8") as json_file:
        json.dump(nlp_record, json_file, indent=4, ensure_ascii=False)

    with open(cleaned_text_path, "w", encoding="utf-8") as text_file:
        text_file.write(processed_text["cleaned_text"])

    nlp_results.append(nlp_record)

print("NLP preprocessing completed:")

for result in nlp_results:
    print(f"- {result['filename']}")
    print(f"  cleaned text: {Path(result['output_paths']['cleaned_text']).name}")
    print(f"  NLP JSON: {Path(result['output_paths']['nlp_json']).name}")

In [ ]:
# create and display a summary of the NLP preprocessing results

nlp_summary_rows = []

for result in nlp_results:
    preprocessing = result["preprocessing"]

    nlp_summary_rows.append({
        "document": result["filename"],
        "best_ocr_source": result["best_ocr_source"],
        "best_ocr_average_confidence": round(result["best_ocr_average_confidence"], 2),
        "raw_character_count": preprocessing["raw_character_count"],
        "normalised_character_count": preprocessing["normalised_character_count"],
        "token_count": preprocessing["token_count"],
        "tokens_without_stopwords_count": preprocessing["tokens_without_stopwords_count"],
        "stem_count": preprocessing["stem_count"],
        "lemma_count": preprocessing["lemma_count"],
        "cleaned_text_preview": preprocessing["cleaned_text"][:150]
    })

nlp_summary_df = pd.DataFrame(nlp_summary_rows)

nlp_summary_csv_path = CSV_DIR / "text_preprocessing_summary.csv"
nlp_summary_df.to_csv(nlp_summary_csv_path, index=False)

nlp_summary_df

In [ ]:
# inspect the NLP preprocessing output for the first document

example_nlp_result = nlp_results[0]
example_preprocessing = example_nlp_result["preprocessing"]

print("Document:", example_nlp_result["filename"])
print("Best OCR source:", example_nlp_result["best_ocr_source"])
print()

print("Raw OCR text preview:")
print("-" * 80)
print(example_preprocessing["normalised_text"][:1000])
print()

print("Tokens:")
print(example_preprocessing["tokens"][:80])
print()

print("Tokens after stopword removal:")
print(example_preprocessing["tokens_without_stopwords"][:80])
print()

print("Stems:")
print(example_preprocessing["stems"][:80])
print()

print("Lemmas:")
print(example_preprocessing["lemmas"][:80])
print()

print("Cleaned text preview:")
print("-" * 80)
print(example_preprocessing["cleaned_text"][:1000])

## Stage 6: NLP Feature Recognition

This stage extracts meaningful textual features from the cleaned OCR text.

The features extracted in this phase are:

1. TF-IDF keywords
2. simple key phrases from cleaned tokens
3. regex-based named entities and document patterns
4. references to tables, figures, diagrams, and signatures

This phase directly supports the feature-recognition part of the rubric. Section categorisation and image-region linking are left for the next phases.

In [ ]:
# Phase 6 NLP feature recognition functions

def deduplicate_preserve_order(values):
    """remove duplicates while preserving the original order"""
    seen = set()
    deduplicated_values = []

    for value in values:
        normalised_value = str(value).strip()

        if normalised_value == "":
            continue

        key = normalised_value.lower()

        if key not in seen:
            seen.add(key)
            deduplicated_values.append(normalised_value)

    return deduplicated_values


def extract_regex_entities(text):
    """extract simple document entities and patterns using regular expressions"""
    text = str(text)

    month_names = (
        "January|February|March|April|May|June|July|August|"
        "September|October|November|December"
    )

    date_patterns = [
        rf"\b\d{{1,2}}\s+(?:{month_names})\s+\d{{4}}\b",
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b"
    ]

    dates = []

    for pattern in date_patterns:
        dates.extend(re.findall(pattern, text, flags=re.IGNORECASE))

    money_values = re.findall(
        r"[€£$]\s*\d+(?:[.,]\d{2})?",
        text
    )

    times = re.findall(
        r"\b\d{1,2}:\d{2}\b",
        text
    )

    urls = re.findall(
        r"\b(?:https?://|www\.)[^\s]+",
        text,
        flags=re.IGNORECASE
    )

    phone_numbers = re.findall(
        r"\b(?:\+?\d[\d\s-]{6,}\d)\b",
        text
    )

    percentages = re.findall(
        r"\b\d+(?:[.,]\d+)?\s*%",
        text
    )

    document_references = re.findall(
        r"\b(?:invoice|receipt|ticket|form|report|notice|ref|reference|id|authorisation|authorization|aid)\s*[:#-]?\s*[A-Z0-9][A-Z0-9_-]*\b",
        text,
        flags=re.IGNORECASE
    )

    return {
        "dates": deduplicate_preserve_order(dates),
        "money_values": deduplicate_preserve_order(money_values),
        "times": deduplicate_preserve_order(times),
        "urls": deduplicate_preserve_order(urls),
        "phone_numbers": deduplicate_preserve_order(phone_numbers),
        "percentages": deduplicate_preserve_order(percentages),
        "document_references": deduplicate_preserve_order(document_references)
    }


def extract_document_reference_terms(text, context_window=45):
    """find references to visual or structural document elements"""
    text = str(text)
    reference_terms = [
        "table",
        "figure",
        "diagram",
        "image",
        "signature",
        "caption",
        "chart"
    ]

    matches = []

    for term in reference_terms:
        for match in re.finditer(rf"\b{term}s?\b", text, flags=re.IGNORECASE):
            start = max(match.start() - context_window, 0)
            end = min(match.end() + context_window, len(text))

            matches.append({
                "term": match.group(0),
                "snippet": text[start:end].strip()
            })

    return matches


def extract_key_phrases_from_tokens(tokens, max_phrases=12):
    """extract simple key phrases using bigrams and trigrams from cleaned tokens"""
    useful_tokens = [
        str(token).lower()
        for token in tokens
        if len(str(token)) > 2 and not str(token).startswith("€")
    ]

    phrase_counter = Counter()

    for ngram_size in [2, 3]:
        for index in range(0, max(len(useful_tokens) - ngram_size + 1, 0)):
            phrase_tokens = useful_tokens[index:index + ngram_size]

            if all(token.replace(".", "").replace(",", "").isdigit() for token in phrase_tokens):
                continue

            phrase = " ".join(phrase_tokens)
            phrase_counter[phrase] += 1

    top_phrases = []

    for phrase, count in phrase_counter.most_common(max_phrases):
        top_phrases.append({
            "phrase": phrase,
            "count": int(count)
        })

    return top_phrases


def build_tfidf_keyword_results(nlp_results, top_n=12):
    """calculate TF-IDF keywords across the available document set"""
    cleaned_texts = [
        result["preprocessing"]["cleaned_text"]
        for result in nlp_results
    ]

    # TF-IDF requires non-empty text, so use a placeholder only for empty OCR failures
    vectorizer_input = [
        text if str(text).strip() else "empty_document_text"
        for text in cleaned_texts
    ]

    vectorizer = TfidfVectorizer(
        max_features=100,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w[\w.-]+\b"
    )

    tfidf_matrix = vectorizer.fit_transform(vectorizer_input)
    feature_names = vectorizer.get_feature_names_out()

    keyword_results = {}

    for row_index, result in enumerate(nlp_results):
        row = tfidf_matrix[row_index].toarray()[0]
        ranked_indexes = row.argsort()[::-1]

        keywords = []

        for feature_index in ranked_indexes[:top_n]:
            score = row[feature_index]

            if score <= 0:
                continue

            keywords.append({
                "keyword": feature_names[feature_index],
                "score": float(score)
            })

        keyword_results[result["filename"]] = keywords

    return keyword_results

In [ ]:
# apply Phase 6 feature recognition to the NLP results

if "nlp_results" not in globals():
    raise NameError("nlp_results does not exist. Run the Phase 5 NLP preprocessing cells before this cell.")

if len(nlp_results) == 0:
    raise ValueError("No NLP results are available for feature recognition.")

tfidf_keyword_lookup = build_tfidf_keyword_results(nlp_results)

text_feature_results = []

for result in nlp_results:
    filename = result["filename"]
    filename_stem = Path(filename).stem
    preprocessing = result["preprocessing"]

    normalised_text = preprocessing["normalised_text"]
    cleaned_tokens = preprocessing["tokens_without_stopwords"]

    regex_entities = extract_regex_entities(normalised_text)
    visual_reference_terms = extract_document_reference_terms(normalised_text)
    key_phrases = extract_key_phrases_from_tokens(cleaned_tokens)

    text_feature_record = {
        "filename": filename,
        "best_ocr_source": result["best_ocr_source"],
        "best_ocr_average_confidence": result["best_ocr_average_confidence"],
        "tfidf_keywords": tfidf_keyword_lookup.get(filename, []),
        "key_phrases": key_phrases,
        "regex_entities": regex_entities,
        "visual_reference_terms": visual_reference_terms
    }

    feature_json_path = TEXT_FEATURE_DIR / f"{filename_stem}_text_features.json"

    with open(feature_json_path, "w", encoding="utf-8") as json_file:
        json.dump(text_feature_record, json_file, indent=4, ensure_ascii=False)

    text_feature_record["output_paths"] = {
        "text_features_json": str(feature_json_path)
    }

    text_feature_results.append(text_feature_record)

print("Text feature recognition completed:")

for result in text_feature_results:
    print(f"- {result['filename']}")
    print(f"  features JSON: {Path(result['output_paths']['text_features_json']).name}")
    print(f"  TF-IDF keywords: {', '.join([item['keyword'] for item in result['tfidf_keywords'][:5]])}")

In [ ]:
# create and display a summary of the text feature recognition results

feature_summary_rows = []

for result in text_feature_results:
    entities = result["regex_entities"]

    feature_summary_rows.append({
        "document": result["filename"],
        "best_ocr_source": result["best_ocr_source"],
        "best_ocr_average_confidence": round(result["best_ocr_average_confidence"], 2),
        "top_tfidf_keywords": ", ".join([item["keyword"] for item in result["tfidf_keywords"][:8]]),
        "top_key_phrases": ", ".join([item["phrase"] for item in result["key_phrases"][:5]]),
        "dates": ", ".join(entities["dates"]),
        "money_values": ", ".join(entities["money_values"]),
        "times": ", ".join(entities["times"]),
        "urls": ", ".join(entities["urls"]),
        "phone_numbers": ", ".join(entities["phone_numbers"]),
        "percentages": ", ".join(entities["percentages"]),
        "document_references": ", ".join(entities["document_references"]),
        "visual_reference_terms_found": len(result["visual_reference_terms"])
    })

text_feature_summary_df = pd.DataFrame(feature_summary_rows)

text_feature_summary_csv_path = CSV_DIR / "text_feature_summary.csv"
text_feature_summary_df.to_csv(text_feature_summary_csv_path, index=False)

text_feature_summary_df

In [ ]:
# inspect Phase 6 feature recognition output for the first document

example_feature_result = text_feature_results[0]

print("Document:", example_feature_result["filename"])
print("Best OCR source:", example_feature_result["best_ocr_source"])
print()

print("Top TF-IDF keywords:")
print(json.dumps(example_feature_result["tfidf_keywords"][:10], indent=4, ensure_ascii=False))
print()

print("Key phrases:")
print(json.dumps(example_feature_result["key_phrases"][:10], indent=4, ensure_ascii=False))
print()

print("Regex-based entities:")
print(json.dumps(example_feature_result["regex_entities"], indent=4, ensure_ascii=False))
print()

print("References to visual/structural document elements:")
print(json.dumps(example_feature_result["visual_reference_terms"], indent=4, ensure_ascii=False))

## Stage 7: Text Analysis and Section Categorisation

This stage analyses the OCR text as document structure rather than as a single block of text.

The section analysis performs four tasks:

1. reconstruct OCR lines from the word-level Tesseract CSV output
2. classify each line into a document section such as header, transaction details, payment details, item/table row, totals, tax summary, visual reference, signature reference, or body text
3. extract simple table-like rows from lines that contain item text and money values
4. identify text references to visual document elements such as tables, figures, diagrams, images, signatures, captions, and charts

This phase supports the rubric requirement for text analysis: categorising sections, extracting tables, and identifying relationships between text and image elements

In [ ]:
# Phase 7 text analysis and section categorisation functions

SECTION_LABEL_ORDER = [
    "header",
    "transaction_details",
    "payment_details",
    "item_table_row",
    "totals",
    "tax_summary",
    "visual_reference",
    "signature_reference",
    "body_text",
    "other"
]


def get_best_ocr_csv_path(ocr_result):
    """return the word-level OCR CSV path for the selected OCR candidate"""
    best_candidate_name = ocr_result["best_candidate_name"]

    for candidate in ocr_result["candidate_results"]:
        if candidate["candidate_name"] == best_candidate_name:
            return Path(candidate["ocr_csv_path"])

    raise ValueError(f"Could not find OCR CSV path for {ocr_result['filename']}.")


def load_best_ocr_words_dataframe(ocr_result):
    """load the word-level OCR dataframe for the selected OCR candidate"""
    csv_path = get_best_ocr_csv_path(ocr_result)

    if not csv_path.exists():
        raise FileNotFoundError(f"OCR CSV file not found: {csv_path}")

    ocr_words = pd.read_csv(csv_path)
    ocr_words["text"] = ocr_words["text"].fillna("").astype(str).str.strip()
    ocr_words = ocr_words[ocr_words["text"] != ""].copy()

    numeric_columns = ["left", "top", "width", "height", "conf"]

    for column in numeric_columns:
        if column in ocr_words.columns:
            ocr_words[column] = pd.to_numeric(ocr_words[column], errors="coerce")

    ocr_words = ocr_words.dropna(subset=["left", "top", "width", "height"])

    return ocr_words


def get_document_height(filename):
    """get the loaded image height for a document filename"""
    for document in loaded_documents:
        if document["filename"] == filename:
            return document["height"]

    return None


def reconstruct_ocr_lines(ocr_words, filename):
    """reconstruct document lines from Tesseract word-level output"""
    group_columns = [
        column for column in ["page_num", "block_num", "par_num", "line_num"]
        if column in ocr_words.columns
    ]

    if len(group_columns) == 0:
        raise ValueError("OCR dataframe does not contain Tesseract line grouping columns.")

    document_height = get_document_height(filename)

    if document_height is None:
        document_height = max(float((ocr_words["top"] + ocr_words["height"]).max()), 1.0)

    line_records = []

    grouped_lines = ocr_words.groupby(group_columns, dropna=False)

    for _, line_dataframe in grouped_lines:
        line_dataframe = line_dataframe.sort_values("left")
        line_text = " ".join(line_dataframe["text"].astype(str).tolist()).strip()

        if line_text == "":
            continue

        x1 = int(line_dataframe["left"].min())
        y1 = int(line_dataframe["top"].min())
        x2 = int((line_dataframe["left"] + line_dataframe["width"]).max())
        y2 = int((line_dataframe["top"] + line_dataframe["height"]).max())

        if "conf" in line_dataframe.columns:
            average_confidence = float(line_dataframe["conf"].mean())
        else:
            average_confidence = 0.0

        line_records.append({
            "text": line_text,
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "top": y1,
            "left": x1,
            "right": x2,
            "bottom": y2,
            "normalised_y": float(y1 / max(document_height, 1)),
            "average_confidence": average_confidence
        })

    line_records = sorted(line_records, key=lambda item: (item["top"], item["left"]))

    for index, line in enumerate(line_records, start=1):
        line["line_index"] = index

    return line_records


def contains_money_value(text):
    """return True when a line contains a currency-style value"""
    return bool(re.search(r"[€£$]\s*\d+(?:[.,]\d{2})?", str(text)))


def extract_money_values_from_text(text):
    """extract currency-style values from a text line"""
    return re.findall(r"[€£$]\s*\d+(?:[.,]\d{2})?", str(text))


def classify_ocr_line(line_text, normalised_y):
    """classify one OCR line into a simple document section label"""
    text = str(line_text).strip()
    lower_text = text.lower()

    if text == "":
        return "other"

    if re.search(r"\b(signature|signed|applicant signature|authorised signature)\b", lower_text):
        return "signature_reference"

    if re.search(r"\b(table|figure|fig\.?|diagram|image|caption|chart)\b", lower_text):
        return "visual_reference"

    if re.search(r"\b(tax|vat|net|rate)\b", lower_text) or re.search(r"\d+(?:[.,]\d+)?\s*%", lower_text):
        return "tax_summary"

    if re.search(r"\b(total|subtotal|balance|amount due|grand total)\b", lower_text):
        return "totals"

    if re.search(r"\b(visa|mastercard|debit|credit|contactless|aid|verified|cash|card|device|payment)\b", lower_text):
        return "payment_details"

    if re.search(r"\b(ticket|receipt|invoice|authorisation|authorization|reference|ref|form id|report id|notice ref)\b", lower_text):
        return "transaction_details"

    if re.search(r"\b\d{1,2}:\d{2}\b", lower_text):
        return "transaction_details"

    month_pattern = (
        "january|february|march|april|may|june|july|august|"
        "september|october|november|december"
    )

    if re.search(rf"\b\d{{1,2}}\s+({month_pattern})\s+\d{{4}}\b", lower_text):
        return "transaction_details"

    if contains_money_value(lower_text):
        return "item_table_row"

    if normalised_y <= 0.25:
        return "header"

    return "body_text"


def categorise_ocr_lines(line_records):
    """categorise reconstructed OCR lines into document sections"""
    categorised_lines = []

    for line in line_records:
        section_label = classify_ocr_line(line["text"], line["normalised_y"])
        categorised_line = dict(line)
        categorised_line["section_label"] = section_label
        categorised_line["money_values"] = extract_money_values_from_text(line["text"])
        categorised_lines.append(categorised_line)

    return categorised_lines


def extract_table_like_rows(categorised_lines):
    """extract simple table-like rows from categorised OCR lines"""
    table_rows = []

    for index, line in enumerate(categorised_lines):
        if line["section_label"] != "item_table_row":
            continue

        money_values = line["money_values"]

        if len(money_values) == 0:
            continue

        source_text = line["text"]
        description = re.sub(r"[€£$]\s*\d+(?:[.,]\d{2})?", "", source_text).strip()
        description = re.sub(r"\s+", " ", description)

        # If OCR split the item name and price across adjacent lines, use the previous non-currency line as context.
        if len(description) <= 2 and index > 0:
            previous_line = categorised_lines[index - 1]

            if len(previous_line["money_values"]) == 0:
                description = previous_line["text"]

        table_rows.append({
            "row_index": len(table_rows) + 1,
            "section_label": line["section_label"],
            "description": description,
            "money_values": money_values,
            "source_text": source_text,
            "line_index": line["line_index"],
            "bbox": line["bbox"]
        })

    return table_rows


def group_text_by_section(categorised_lines):
    """group OCR line text by section label"""
    grouped_text = {label: [] for label in SECTION_LABEL_ORDER}

    for line in categorised_lines:
        label = line["section_label"]

        if label not in grouped_text:
            grouped_text[label] = []

        grouped_text[label].append(line["text"])

    return grouped_text


def count_sections(categorised_lines):
    """count how many OCR lines were assigned to each section"""
    counts = {label: 0 for label in SECTION_LABEL_ORDER}

    for line in categorised_lines:
        label = line["section_label"]
        counts[label] = counts.get(label, 0) + 1

    return counts


def build_text_image_reference_records(text_feature_result, categorised_lines):
    """build simple text-image relationship records from text references"""
    reference_records = []

    for reference in text_feature_result.get("visual_reference_terms", []):
        reference_records.append({
            "relationship_type": "textual_visual_reference",
            "term": reference.get("term", ""),
            "snippet": reference.get("snippet", ""),
            "linked_visual_region": "pending visual detection phase"
        })

    for line in categorised_lines:
        if line["section_label"] in ["visual_reference", "signature_reference"]:
            reference_records.append({
                "relationship_type": line["section_label"],
                "term": line["section_label"],
                "snippet": line["text"],
                "linked_visual_region": "pending visual detection phase"
            })

    return reference_records

In [ ]:
# apply Phase 7 section analysis to each document

if "ocr_results" not in globals():
    raise NameError("ocr_results does not exist. Run the OCR cells before this cell.")

if "text_feature_results" not in globals():
    raise NameError("text_feature_results does not exist. Run the Phase 6 text feature cells before this cell.")

text_feature_lookup = {
    result["filename"]: result
    for result in text_feature_results
}

section_analysis_results = []

for ocr_result in ocr_results:
    filename = ocr_result["filename"]
    filename_stem = Path(filename).stem

    ocr_words = load_best_ocr_words_dataframe(ocr_result)
    reconstructed_lines = reconstruct_ocr_lines(ocr_words, filename)
    categorised_lines = categorise_ocr_lines(reconstructed_lines)
    grouped_text = group_text_by_section(categorised_lines)
    section_counts = count_sections(categorised_lines)
    table_like_rows = extract_table_like_rows(categorised_lines)

    text_feature_result = text_feature_lookup.get(filename, {})
    text_image_reference_records = build_text_image_reference_records(
        text_feature_result,
        categorised_lines
    )

    section_record = {
        "filename": filename,
        "best_ocr_source": ocr_result["best_candidate_name"],
        "best_ocr_average_confidence": ocr_result["best_average_confidence"],
        "line_count": len(categorised_lines),
        "section_counts": section_counts,
        "grouped_text_by_section": grouped_text,
        "categorised_lines": categorised_lines,
        "table_like_rows": table_like_rows,
        "text_image_reference_records": text_image_reference_records
    }

    section_json_path = SECTION_ANALYSIS_DIR / f"{filename_stem}_section_analysis.json"

    with open(section_json_path, "w", encoding="utf-8") as json_file:
        json.dump(section_record, json_file, indent=4, ensure_ascii=False)

    section_record["output_paths"] = {
        "section_analysis_json": str(section_json_path)
    }

    section_analysis_results.append(section_record)

print("Section analysis completed:")

for result in section_analysis_results:
    print(f"- {result['filename']}")
    print(f"  lines categorised: {result['line_count']}")
    print(f"  table-like rows: {len(result['table_like_rows'])}")
    print(f"  text-image references: {len(result['text_image_reference_records'])}")
    print(f"  section JSON: {Path(result['output_paths']['section_analysis_json']).name}")

In [ ]:
# create and display Phase 7 section analysis summary tables

section_summary_rows = []
section_line_rows = []
table_row_records = []

for result in section_analysis_results:
    counts = result["section_counts"]

    summary_row = {
        "document": result["filename"],
        "best_ocr_source": result["best_ocr_source"],
        "best_ocr_average_confidence": round(result["best_ocr_average_confidence"], 2),
        "line_count": result["line_count"],
        "table_like_row_count": len(result["table_like_rows"]),
        "text_image_reference_count": len(result["text_image_reference_records"])
    }

    for label in SECTION_LABEL_ORDER:
        summary_row[f"{label}_lines"] = counts.get(label, 0)

    section_summary_rows.append(summary_row)

    for line in result["categorised_lines"]:
        section_line_rows.append({
            "document": result["filename"],
            "line_index": line["line_index"],
            "section_label": line["section_label"],
            "text": line["text"],
            "bbox": line["bbox"],
            "average_confidence": round(line["average_confidence"], 2)
        })

    for table_row in result["table_like_rows"]:
        table_row_records.append({
            "document": result["filename"],
            "row_index": table_row["row_index"],
            "description": table_row["description"],
            "money_values": ", ".join(table_row["money_values"]),
            "source_text": table_row["source_text"],
            "line_index": table_row["line_index"]
        })

section_analysis_summary_df = pd.DataFrame(section_summary_rows)
section_lines_df = pd.DataFrame(section_line_rows)
extracted_table_rows_df = pd.DataFrame(table_row_records)

section_analysis_summary_csv_path = CSV_DIR / "section_analysis_summary.csv"
section_lines_csv_path = CSV_DIR / "section_lines.csv"
extracted_table_rows_csv_path = CSV_DIR / "extracted_table_rows.csv"

section_analysis_summary_df.to_csv(section_analysis_summary_csv_path, index=False)
section_lines_df.to_csv(section_lines_csv_path, index=False)
extracted_table_rows_df.to_csv(extracted_table_rows_csv_path, index=False)

section_analysis_summary_df

In [ ]:
# inspect Phase 7 section categorisation for the first document

example_section_result = section_analysis_results[0]

print("Document:", example_section_result["filename"])
print("Best OCR source:", example_section_result["best_ocr_source"])
print("Section counts:")
print(json.dumps(example_section_result["section_counts"], indent=4, ensure_ascii=False))
print()

print("Grouped text by section:")
for section_label in SECTION_LABEL_ORDER:
    section_lines = example_section_result["grouped_text_by_section"].get(section_label, [])

    if len(section_lines) == 0:
        continue

    print(f"\n[{section_label}]")

    for line in section_lines[:8]:
        print("-", line)

print("\nExtracted table-like rows:")
print(json.dumps(example_section_result["table_like_rows"], indent=4, ensure_ascii=False))
print()

print("Text-image relationship records:")
print(json.dumps(example_section_result["text_image_reference_records"], indent=4, ensure_ascii=False))

### Phase 7 notes

The section labels are rule-based and depend on the quality of the OCR text. This is intentional for this development phase: the notebook now demonstrates document text analysis without adding visual feature detection yet.

For the receipt test image, the expected useful sections are header, transaction details, payment details, item/table rows, totals, and tax summary. Some lines may be misclassified when OCR introduces spelling errors or splits related text across separate lines.


## Stage 8: OpenCV Visual Feature Detection

This stage adds visual feature detection using the preprocessed document images and the OCR layout data.

The aim is to detect and label visible document features rather than only analysing the extracted text.

The detection logic looks for:

1. horizontal separator lines
2. table-like or grid-like regions
3. OCR text-block regions
4. signature-like regions
5. diagram or figure-like candidate regions

The output from this stage is an annotated image plus JSON and CSV records describing the detected visual features.

This phase supports the rubric requirement for feature detection: identifying tables, signatures, diagrams, figures, and other document layout elements.


In [ ]:
# Phase 8 OpenCV visual feature detection functions

VISUAL_FEATURE_COLOURS = {
    "horizontal_separator_line": (255, 0, 0),
    "table_like_region": (0, 180, 0),
    "table_grid_region": (0, 120, 0),
    "text_block_region": (0, 0, 255),
    "signature_like_region": (180, 0, 180),
    "diagram_or_figure_candidate": (0, 160, 220),
    "document_boundary_candidate": (120, 120, 120)
}


def get_loaded_document_by_filename(filename):
    """return the loaded document record for a filename"""
    for document in loaded_documents:
        if document["filename"] == filename:
            return document

    raise ValueError(f"Loaded document not found: {filename}")


def get_preprocessed_document_by_filename(filename):
    """return the preprocessed document record for a filename"""
    for document in preprocessed_documents:
        if document["filename"] == filename:
            return document

    raise ValueError(f"Preprocessed document not found: {filename}")


def get_best_threshold_image(preprocessed_document):
    """return the best available binary image for visual feature detection"""
    processed_images = preprocessed_document["processed_images"]

    preferred_names = [
        "adaptive_threshold_clean",
        "adaptive_threshold",
        "otsu_threshold"
    ]

    for name in preferred_names:
        if name in processed_images:
            return processed_images[name], name

    raise ValueError(f"No threshold image found for {preprocessed_document['filename']}.")


def get_feature_base_edges(preprocessed_document):
    """return the edge image generated in the preprocessing phase"""
    processed_images = preprocessed_document["processed_images"]

    if "edges" not in processed_images:
        raise ValueError(f"No edge image found for {preprocessed_document['filename']}.")

    return processed_images["edges"]


def normalise_bbox(x, y, w, h):
    """return a bbox as normal Python integers"""
    return [int(x), int(y), int(w), int(h)]


def count_ocr_words_inside_bbox(ocr_words, bbox):
    """count OCR words whose centres fall inside a bounding box"""
    if ocr_words is None or len(ocr_words) == 0:
        return 0

    x, y, w, h = bbox
    x2 = x + w
    y2 = y + h

    word_centres_x = ocr_words["left"] + (ocr_words["width"] / 2)
    word_centres_y = ocr_words["top"] + (ocr_words["height"] / 2)

    words_inside = ocr_words[
        (word_centres_x >= x) &
        (word_centres_x <= x2) &
        (word_centres_y >= y) &
        (word_centres_y <= y2)
    ]

    return int(len(words_inside))


def combine_line_bboxes(lines):
    """combine OCR line bounding boxes into one larger bounding box"""
    if len(lines) == 0:
        return None

    left_values = []
    top_values = []
    right_values = []
    bottom_values = []

    for line in lines:
        x, y, w, h = line["bbox"]
        left_values.append(x)
        top_values.append(y)
        right_values.append(x + w)
        bottom_values.append(y + h)

    x1 = min(left_values)
    y1 = min(top_values)
    x2 = max(right_values)
    y2 = max(bottom_values)

    return normalise_bbox(x1, y1, x2 - x1, y2 - y1)


def merge_close_horizontal_lines(line_features, y_tolerance=8):
    """merge duplicate horizontal line detections that are close together"""
    if len(line_features) == 0:
        return []

    sorted_lines = sorted(
        line_features,
        key=lambda feature: (feature["bbox"][1], -feature["bbox"][2])
    )

    merged = []

    for feature in sorted_lines:
        x, y, w, h = feature["bbox"]

        if len(merged) == 0:
            merged.append(feature)
            continue

        last = merged[-1]
        last_x, last_y, last_w, last_h = last["bbox"]

        if abs(y - last_y) <= y_tolerance:
            if w > last_w:
                merged[-1] = feature
        else:
            merged.append(feature)

    return merged


def detect_horizontal_separator_lines(threshold_image):
    """detect long horizontal lines using morphology and contours"""
    image_height, image_width = threshold_image.shape[:2]

    inverted = 255 - threshold_image

    kernel_width = max(40, int(image_width * 0.20))
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_width, 1))

    horizontal_mask = cv2.morphologyEx(
        inverted,
        cv2.MORPH_OPEN,
        horizontal_kernel
    )

    contours, _ = cv2.findContours(
        horizontal_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    line_features = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)

        if w < image_width * 0.25:
            continue

        if h > 20:
            continue

        line_features.append({
            "feature_type": "horizontal_separator_line",
            "bbox": normalise_bbox(x, y, w, h),
            "detection_method": "morphological horizontal line extraction",
            "confidence_note": "rule_based_candidate"
        })

    return merge_close_horizontal_lines(line_features)


def detect_grid_table_regions(threshold_image, ocr_words):
    """detect grid-like table regions from horizontal and vertical line masks"""
    image_height, image_width = threshold_image.shape[:2]
    inverted = 255 - threshold_image

    horizontal_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (max(40, int(image_width * 0.12)), 1)
    )

    vertical_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (1, max(40, int(image_height * 0.04)))
    )

    horizontal_lines = cv2.morphologyEx(inverted, cv2.MORPH_OPEN, horizontal_kernel)
    vertical_lines = cv2.morphologyEx(inverted, cv2.MORPH_OPEN, vertical_kernel)

    grid_mask = cv2.add(horizontal_lines, vertical_lines)
    grid_mask = cv2.dilate(
        grid_mask,
        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)),
        iterations=1
    )

    contours, _ = cv2.findContours(
        grid_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    table_features = []

    image_area = image_width * image_height

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h

        if area < image_area * 0.01:
            continue

        if w < image_width * 0.25 or h < image_height * 0.05:
            continue

        bbox = normalise_bbox(x, y, w, h)

        table_features.append({
            "feature_type": "table_grid_region",
            "bbox": bbox,
            "detection_method": "horizontal and vertical morphology",
            "ocr_words_inside": count_ocr_words_inside_bbox(ocr_words, bbox),
            "confidence_note": "rule_based_candidate"
        })

    return table_features


def detect_table_like_regions_from_separator_lines(horizontal_lines, ocr_words):
    """detect table-like regions between horizontal separator lines"""
    if len(horizontal_lines) < 2:
        return []

    sorted_lines = sorted(horizontal_lines, key=lambda feature: feature["bbox"][1])

    table_like_regions = []

    for index in range(len(sorted_lines) - 1):
        upper_line = sorted_lines[index]
        lower_line = sorted_lines[index + 1]

        upper_x, upper_y, upper_w, upper_h = upper_line["bbox"]
        lower_x, lower_y, lower_w, lower_h = lower_line["bbox"]

        region_top = upper_y + upper_h
        region_bottom = lower_y
        region_height = region_bottom - region_top

        if region_height < 35:
            continue

        region_left = min(upper_x, lower_x)
        region_right = max(upper_x + upper_w, lower_x + lower_w)
        region_width = region_right - region_left

        bbox = normalise_bbox(region_left, region_top, region_width, region_height)
        words_inside = count_ocr_words_inside_bbox(ocr_words, bbox)

        if words_inside == 0:
            continue

        table_like_regions.append({
            "feature_type": "table_like_region",
            "bbox": bbox,
            "detection_method": "region between horizontal separator lines",
            "ocr_words_inside": words_inside,
            "confidence_note": "rule_based_candidate"
        })

    return table_like_regions


def detect_text_block_regions_from_ocr(section_analysis_result):
    """create visual text-block regions from Phase 7 OCR line groupings"""
    text_block_features = []

    categorised_lines = section_analysis_result["categorised_lines"]

    for section_label in SECTION_LABEL_ORDER:
        section_lines = [
            line for line in categorised_lines
            if line["section_label"] == section_label
        ]

        if len(section_lines) == 0:
            continue

        bbox = combine_line_bboxes(section_lines)

        if bbox is None:
            continue

        text_block_features.append({
            "feature_type": "text_block_region",
            "section_label": section_label,
            "bbox": bbox,
            "detection_method": "OCR line grouping from Tesseract layout",
            "line_count": len(section_lines),
            "confidence_note": "ocr_layout_based"
        })

    return text_block_features


def detect_signature_like_regions(threshold_image, ocr_words):
    """detect signature-like regions using lower-page connected components"""
    image_height, image_width = threshold_image.shape[:2]

    lower_start_y = int(image_height * 0.55)
    lower_region = threshold_image[lower_start_y:image_height, :]

    inverted = 255 - lower_region

    contours, _ = cv2.findContours(
        inverted,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    signature_features = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        y_absolute = y + lower_start_y
        area = w * h

        if area < 500 or area > image_width * image_height * 0.04:
            continue

        if w < 80 or h < 18 or h > 160:
            continue

        aspect_ratio = w / max(h, 1)

        if aspect_ratio < 2.0:
            continue

        bbox = normalise_bbox(x, y_absolute, w, h)
        words_inside = count_ocr_words_inside_bbox(ocr_words, bbox)

        if words_inside > 5:
            continue

        signature_features.append({
            "feature_type": "signature_like_region",
            "bbox": bbox,
            "detection_method": "lower-page connected component shape filtering",
            "ocr_words_inside": words_inside,
            "confidence_note": "rule_based_candidate"
        })

    return signature_features


def detect_diagram_or_figure_candidates(edge_image, ocr_words):
    """detect large non-text regions that may be diagrams or figures"""
    image_height, image_width = edge_image.shape[:2]
    image_area = image_width * image_height

    closed_edges = cv2.morphologyEx(
        edge_image,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    )

    contours, _ = cv2.findContours(
        closed_edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    figure_features = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h

        if area < image_area * 0.015:
            continue

        if area > image_area * 0.45:
            continue

        if w < image_width * 0.15 or h < image_height * 0.07:
            continue

        bbox = normalise_bbox(x, y, w, h)
        words_inside = count_ocr_words_inside_bbox(ocr_words, bbox)

        if words_inside > 8:
            continue

        figure_features.append({
            "feature_type": "diagram_or_figure_candidate",
            "bbox": bbox,
            "detection_method": "large closed edge component with low OCR word density",
            "ocr_words_inside": words_inside,
            "confidence_note": "rule_based_candidate"
        })

    return figure_features


def draw_visual_feature_annotations(image, feature_records):
    """draw detected visual feature boxes on a copy of the original image"""
    annotated = image.copy()

    for feature in feature_records:
        x, y, w, h = feature["bbox"]
        feature_type = feature["feature_type"]

        colour = VISUAL_FEATURE_COLOURS.get(feature_type, (0, 0, 255))

        cv2.rectangle(
            annotated,
            (x, y),
            (x + w, y + h),
            colour,
            2
        )

        if feature_type == "text_block_region":
            label = feature.get("section_label", "text_block")
        else:
            label = feature_type.replace("_", " ")

        cv2.putText(
            annotated,
            label[:32],
            (x, max(y - 8, 18)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            colour,
            2,
            cv2.LINE_AA
        )

    return annotated

In [ ]:
# apply Phase 8 visual feature detection to each document

if "preprocessed_documents" not in globals():
    raise NameError("preprocessed_documents does not exist. Run the preprocessing cells before this cell.")

if "section_analysis_results" not in globals():
    raise NameError("section_analysis_results does not exist. Run the Phase 7 section analysis cells before this cell.")

visual_feature_results = []

for section_result in section_analysis_results:
    filename = section_result["filename"]
    filename_stem = Path(filename).stem

    loaded_document = get_loaded_document_by_filename(filename)
    preprocessed_document = get_preprocessed_document_by_filename(filename)

    threshold_image, threshold_source = get_best_threshold_image(preprocessed_document)
    edge_image = get_feature_base_edges(preprocessed_document)
    matching_ocr_result = next(result for result in ocr_results if result["filename"] == filename)
    ocr_words = load_best_ocr_words_dataframe(matching_ocr_result)

    horizontal_lines = detect_horizontal_separator_lines(threshold_image)
    table_grid_regions = detect_grid_table_regions(threshold_image, ocr_words)
    table_like_regions = detect_table_like_regions_from_separator_lines(horizontal_lines, ocr_words)
    text_block_regions = detect_text_block_regions_from_ocr(section_result)
    signature_like_regions = detect_signature_like_regions(threshold_image, ocr_words)
    figure_like_regions = detect_diagram_or_figure_candidates(edge_image, ocr_words)

    all_features = []
    all_features.extend(horizontal_lines)
    all_features.extend(table_grid_regions)
    all_features.extend(table_like_regions)
    all_features.extend(text_block_regions)
    all_features.extend(signature_like_regions)
    all_features.extend(figure_like_regions)

    annotated_image = draw_visual_feature_annotations(
        loaded_document["image"],
        all_features
    )

    annotated_image_path = ANNOTATED_DIR / f"{filename_stem}_visual_features.png"
    visual_feature_json_path = VISUAL_FEATURE_DIR / f"{filename_stem}_visual_features.json"

    cv2.imwrite(str(annotated_image_path), annotated_image)

    feature_counts = {}

    for feature in all_features:
        feature_type = feature["feature_type"]
        feature_counts[feature_type] = feature_counts.get(feature_type, 0) + 1

    visual_feature_record = {
        "filename": filename,
        "threshold_source": threshold_source,
        "feature_counts": feature_counts,
        "feature_records": all_features,
        "output_paths": {
            "annotated_image": str(annotated_image_path),
            "visual_feature_json": str(visual_feature_json_path)
        }
    }

    with open(visual_feature_json_path, "w", encoding="utf-8") as json_file:
        json.dump(visual_feature_record, json_file, indent=4, ensure_ascii=False)

    visual_feature_results.append(visual_feature_record)

print("Visual feature detection completed:")

for result in visual_feature_results:
    print(f"- {result['filename']}")
    print(f"  threshold source: {result['threshold_source']}")
    print(f"  annotated image: {Path(result['output_paths']['annotated_image']).name}")

    for feature_type, count in result["feature_counts"].items():
        print(f"  {feature_type}: {count}")

In [ ]:
# create and display Phase 8 visual feature summary tables

visual_feature_summary_rows = []
visual_feature_region_rows = []

for result in visual_feature_results:
    feature_counts = result["feature_counts"]

    summary_row = {
        "document": result["filename"],
        "threshold_source": result["threshold_source"],
        "total_visual_features": len(result["feature_records"]),
        "horizontal_separator_lines": feature_counts.get("horizontal_separator_line", 0),
        "table_grid_regions": feature_counts.get("table_grid_region", 0),
        "table_like_regions": feature_counts.get("table_like_region", 0),
        "text_block_regions": feature_counts.get("text_block_region", 0),
        "signature_like_regions": feature_counts.get("signature_like_region", 0),
        "diagram_or_figure_candidates": feature_counts.get("diagram_or_figure_candidate", 0),
        "annotated_image": result["output_paths"]["annotated_image"],
        "visual_feature_json": result["output_paths"]["visual_feature_json"]
    }

    visual_feature_summary_rows.append(summary_row)

    for feature_index, feature in enumerate(result["feature_records"], start=1):
        visual_feature_region_rows.append({
            "document": result["filename"],
            "feature_index": feature_index,
            "feature_type": feature["feature_type"],
            "section_label": feature.get("section_label", ""),
            "bbox": feature["bbox"],
            "detection_method": feature.get("detection_method", ""),
            "ocr_words_inside": feature.get("ocr_words_inside", ""),
            "confidence_note": feature.get("confidence_note", "")
        })

visual_feature_summary_df = pd.DataFrame(visual_feature_summary_rows)
visual_feature_regions_df = pd.DataFrame(visual_feature_region_rows)

visual_feature_summary_csv_path = CSV_DIR / "visual_feature_summary.csv"
visual_feature_regions_csv_path = CSV_DIR / "visual_feature_regions.csv"

visual_feature_summary_df.to_csv(visual_feature_summary_csv_path, index=False)
visual_feature_regions_df.to_csv(visual_feature_regions_csv_path, index=False)

visual_feature_summary_df

In [ ]:
# inspect the visual feature annotation for the first document

example_visual_feature_result = visual_feature_results[0]

annotated_image_path = Path(example_visual_feature_result["output_paths"]["annotated_image"])
annotated_image = cv2.imread(str(annotated_image_path))

if annotated_image is None:
    raise FileNotFoundError(f"Could not load annotated image: {annotated_image_path}")

annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 14))
plt.imshow(annotated_image_rgb)
plt.axis("off")
plt.title(f"Phase 8 visual feature detection: {example_visual_feature_result['filename']}")
plt.show()

print(json.dumps(example_visual_feature_result["feature_counts"], indent=4))

### Phase 8 notes

The visual feature detection is rule-based rather than machine-learning based.

This is intentional for this assignment stage because the features are explainable:

- horizontal separator lines are found using morphology
- table-like regions are inferred from separator lines and grid-like line structures
- text blocks are created from OCR line groupings
- signature-like candidates are found using lower-page connected components
- diagram or figure candidates are found as large low-text edge regions

The method is expected to produce some false positives on photographed documents because folds, shadows, and background texture can create strong visual edges.

## Stage 9: Image Segmentation into Labelled Regions

This stage converts the visual feature detections and OCR section analysis into labelled document regions.

The segmentation approach is rule-based and uses evidence already produced by earlier stages:

1. OCR line positions from the text analysis stage
2. visual feature detections from OpenCV
3. horizontal separator lines detected in the document image
4. document image dimensions

The output is a set of labelled regions such as header, transaction details, payment details, item/table rows, totals, tax summary, table-like regions, signature-like regions, and diagram/figure candidates.

This stage is not intended to be a perfect production-grade segmentation algorithm. Its purpose is to show meaningful segmentation of document regions for the CA2 rubric.

In [ ]:
# Phase 9 image segmentation functions

SEGMENTATION_COLOURS = {
    "text_section": (0, 0, 255),
    "visual_feature": (0, 180, 0),
    "horizontal_band": (255, 0, 0),
    "table_region": (0, 120, 0),
    "signature_region": (180, 0, 180),
    "diagram_region": (0, 160, 220)
}


def get_visual_feature_result_by_filename(filename):
    """return the visual feature record for a filename"""
    for result in visual_feature_results:
        if result["filename"] == filename:
            return result

    raise ValueError(f"Visual feature result not found: {filename}")


def get_loaded_image_shape(filename):
    """return the loaded image shape for a filename"""
    loaded_document = get_loaded_document_by_filename(filename)
    return loaded_document["image"].shape


def clip_bbox_to_image(bbox, image_shape):
    """clip a bounding box so it stays inside the document image"""
    image_height, image_width = image_shape[:2]

    x, y, w, h = bbox

    x = max(0, int(x))
    y = max(0, int(y))
    w = max(1, int(w))
    h = max(1, int(h))

    x2 = min(image_width, x + w)
    y2 = min(image_height, y + h)

    return [x, y, max(1, x2 - x), max(1, y2 - y)]


def combine_bboxes(bboxes):
    """combine multiple bounding boxes into one larger bounding box"""
    if len(bboxes) == 0:
        return None

    left_values = []
    top_values = []
    right_values = []
    bottom_values = []

    for bbox in bboxes:
        x, y, w, h = bbox
        left_values.append(x)
        top_values.append(y)
        right_values.append(x + w)
        bottom_values.append(y + h)

    x1 = min(left_values)
    y1 = min(top_values)
    x2 = max(right_values)
    y2 = max(bottom_values)

    return [int(x1), int(y1), int(x2 - x1), int(y2 - y1)]


def bbox_area(bbox):
    """return bounding box area"""
    _, _, w, h = bbox
    return int(max(0, w) * max(0, h))


def get_lines_inside_bbox(categorised_lines, bbox):
    """return OCR lines whose centre point falls inside a bounding box"""
    x, y, w, h = bbox
    x2 = x + w
    y2 = y + h

    lines_inside = []

    for line in categorised_lines:
        lx, ly, lw, lh = line["bbox"]
        centre_x = lx + (lw / 2)
        centre_y = ly + (lh / 2)

        if x <= centre_x <= x2 and y <= centre_y <= y2:
            lines_inside.append(line)

    return lines_inside


def create_text_section_segments(section_result, image_shape):
    """create region segments by combining OCR line boxes for each section label"""
    categorised_lines = section_result["categorised_lines"]
    section_groups = {}

    for line in categorised_lines:
        label = line.get("section_label", "other")

        if label not in section_groups:
            section_groups[label] = []

        section_groups[label].append(line)

    segments = []

    for section_label, lines in section_groups.items():
        line_bboxes = [line["bbox"] for line in lines]
        combined_bbox = combine_bboxes(line_bboxes)

        if combined_bbox is None:
            continue

        combined_bbox = clip_bbox_to_image(combined_bbox, image_shape)

        if bbox_area(combined_bbox) < 100:
            continue

        preview_text = " ".join([line["text"] for line in lines])[:250]

        segments.append({
            "segment_type": "text_section",
            "segment_label": section_label,
            "source": "phase_7_ocr_section_analysis",
            "bbox": combined_bbox,
            "ocr_line_count": len(lines),
            "text_preview": preview_text,
            "detection_method": "combined OCR line bounding boxes by section label",
            "confidence_note": "rule_based_segmentation"
        })

    return segments


def map_feature_type_to_segment_label(feature_type):
    """map Phase 8 visual feature types to Phase 9 segment labels"""
    if feature_type in ["table_like_region", "table_grid_region"]:
        return "table_region"

    if feature_type == "signature_like_region":
        return "signature_region"

    if feature_type == "diagram_or_figure_candidate":
        return "diagram_or_figure_region"

    if feature_type == "text_block_region":
        return "text_block_region"

    return feature_type


def create_visual_feature_segments(visual_feature_result, section_result, image_shape):
    """create segmentation records from the Phase 8 visual features"""
    categorised_lines = section_result["categorised_lines"]
    visual_segments = []

    useful_feature_types = {
        "table_like_region",
        "table_grid_region",
        "text_block_region",
        "signature_like_region",
        "diagram_or_figure_candidate"
    }

    for feature in visual_feature_result["feature_records"]:
        feature_type = feature["feature_type"]

        if feature_type not in useful_feature_types:
            continue

        bbox = clip_bbox_to_image(feature["bbox"], image_shape)

        if bbox_area(bbox) < 250:
            continue

        lines_inside = get_lines_inside_bbox(categorised_lines, bbox)
        preview_text = " ".join([line["text"] for line in lines_inside])[:250]

        segment_label = map_feature_type_to_segment_label(feature_type)

        segment_type = "visual_feature"

        if segment_label == "table_region":
            segment_type = "table_region"
        elif segment_label == "signature_region":
            segment_type = "signature_region"
        elif segment_label == "diagram_or_figure_region":
            segment_type = "diagram_region"

        visual_segments.append({
            "segment_type": segment_type,
            "segment_label": segment_label,
            "source": "phase_8_visual_feature_detection",
            "bbox": bbox,
            "ocr_line_count": len(lines_inside),
            "ocr_words_inside": feature.get("ocr_words_inside", ""),
            "text_preview": preview_text,
            "detection_method": feature.get("detection_method", ""),
            "confidence_note": feature.get("confidence_note", "rule_based_candidate")
        })

    return visual_segments


def create_horizontal_band_segments(visual_feature_result, section_result, image_shape, minimum_band_height=50):
    """create broad horizontal document bands from detected separator lines"""
    image_height, image_width = image_shape[:2]
    categorised_lines = section_result["categorised_lines"]

    separator_y_values = []

    for feature in visual_feature_result["feature_records"]:
        if feature["feature_type"] != "horizontal_separator_line":
            continue

        x, y, w, h = feature["bbox"]
        separator_y_values.append(int(y + (h / 2)))

    separator_y_values = sorted(set(separator_y_values))

    if len(separator_y_values) == 0:
        return []

    boundaries = [0] + separator_y_values + [image_height]
    band_segments = []

    for index in range(len(boundaries) - 1):
        y1 = boundaries[index]
        y2 = boundaries[index + 1]

        if y2 - y1 < minimum_band_height:
            continue

        bbox = [0, y1, image_width, y2 - y1]
        lines_inside = get_lines_inside_bbox(categorised_lines, bbox)

        if len(lines_inside) == 0:
            continue

        label_counter = Counter([line["section_label"] for line in lines_inside])
        dominant_label = label_counter.most_common(1)[0][0]
        preview_text = " ".join([line["text"] for line in lines_inside])[:250]

        band_segments.append({
            "segment_type": "horizontal_band",
            "segment_label": f"band_{dominant_label}",
            "source": "phase_8_horizontal_separator_lines",
            "bbox": bbox,
            "ocr_line_count": len(lines_inside),
            "text_preview": preview_text,
            "detection_method": "document bands created from detected horizontal separator lines",
            "confidence_note": "broad_region_candidate"
        })

    return band_segments


def assign_segment_ids(segments):
    """sort and assign stable segment IDs"""
    sorted_segments = sorted(
        segments,
        key=lambda segment: (
            segment["bbox"][1],
            segment["bbox"][0],
            segment["segment_type"],
            segment["segment_label"]
        )
    )

    for index, segment in enumerate(sorted_segments, start=1):
        segment["segment_id"] = f"S{index:03d}"

    return sorted_segments


def get_segment_colour(segment):
    """choose a colour for a segment based on segment type"""
    segment_type = segment.get("segment_type", "text_section")
    return SEGMENTATION_COLOURS.get(segment_type, (80, 80, 80))


def draw_segmentation_annotations(image, segments):
    """draw labelled segmentation boxes on a document image"""
    annotated = image.copy()

    for segment in segments:
        x, y, w, h = segment["bbox"]
        colour = get_segment_colour(segment)
        label = f"{segment['segment_id']} {segment['segment_label']}"

        cv2.rectangle(
            annotated,
            (x, y),
            (x + w, y + h),
            colour,
            2
        )

        label_y = max(y - 8, 20)

        cv2.putText(
            annotated,
            label,
            (x + 5, label_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            colour,
            1,
            cv2.LINE_AA
        )

    return annotated

In [ ]:
# apply Phase 9 segmentation to each document

if "section_analysis_results" not in globals():
    raise NameError("section_analysis_results does not exist. Run the Phase 7 section analysis cells before this cell.")

if "visual_feature_results" not in globals():
    raise NameError("visual_feature_results does not exist. Run the Phase 8 visual feature detection cells before this cell.")

segmentation_results = []

for section_result in section_analysis_results:
    filename = section_result["filename"]
    filename_stem = Path(filename).stem

    loaded_document = get_loaded_document_by_filename(filename)
    visual_feature_result = get_visual_feature_result_by_filename(filename)
    image_shape = get_loaded_image_shape(filename)

    text_section_segments = create_text_section_segments(section_result, image_shape)
    visual_feature_segments = create_visual_feature_segments(
        visual_feature_result,
        section_result,
        image_shape
    )
    horizontal_band_segments = create_horizontal_band_segments(
        visual_feature_result,
        section_result,
        image_shape
    )

    all_segments = []
    all_segments.extend(horizontal_band_segments)
    all_segments.extend(text_section_segments)
    all_segments.extend(visual_feature_segments)

    all_segments = assign_segment_ids(all_segments)

    annotated_segmentation_image = draw_segmentation_annotations(
        loaded_document["image"],
        all_segments
    )

    segmentation_image_path = ANNOTATED_DIR / f"{filename_stem}_segmented_regions.png"
    segmentation_json_path = SEGMENTATION_DIR / f"{filename_stem}_segmentation.json"

    cv2.imwrite(str(segmentation_image_path), annotated_segmentation_image)

    segment_counts = {}

    for segment in all_segments:
        segment_type = segment["segment_type"]
        segment_counts[segment_type] = segment_counts.get(segment_type, 0) + 1

    segmentation_record = {
        "filename": filename,
        "segment_count": len(all_segments),
        "segment_counts": segment_counts,
        "segments": all_segments,
        "output_paths": {
            "segmentation_json": str(segmentation_json_path),
            "annotated_segmentation_image": str(segmentation_image_path)
        }
    }

    with open(segmentation_json_path, "w", encoding="utf-8") as json_file:
        json.dump(segmentation_record, json_file, indent=4, ensure_ascii=False)

    segmentation_results.append(segmentation_record)

print("Image segmentation completed:")

for result in segmentation_results:
    print(f"- {result['filename']}")
    print(f"  segments: {result['segment_count']}")
    print(f"  annotated image: {Path(result['output_paths']['annotated_segmentation_image']).name}")

    for segment_type, count in result["segment_counts"].items():
        print(f"  {segment_type}: {count}")

In [ ]:
# create and display Phase 9 segmentation summary tables

segmentation_summary_rows = []
segmented_region_rows = []

for result in segmentation_results:
    segment_counts = result["segment_counts"]

    segmentation_summary_rows.append({
        "document": result["filename"],
        "total_segments": result["segment_count"],
        "horizontal_band_segments": segment_counts.get("horizontal_band", 0),
        "text_section_segments": segment_counts.get("text_section", 0),
        "table_region_segments": segment_counts.get("table_region", 0),
        "signature_region_segments": segment_counts.get("signature_region", 0),
        "diagram_region_segments": segment_counts.get("diagram_region", 0),
        "visual_feature_segments": segment_counts.get("visual_feature", 0),
        "annotated_segmentation_image": result["output_paths"]["annotated_segmentation_image"],
        "segmentation_json": result["output_paths"]["segmentation_json"]
    })

    for segment in result["segments"]:
        segmented_region_rows.append({
            "document": result["filename"],
            "segment_id": segment["segment_id"],
            "segment_type": segment["segment_type"],
            "segment_label": segment["segment_label"],
            "source": segment["source"],
            "bbox": segment["bbox"],
            "ocr_line_count": segment.get("ocr_line_count", 0),
            "ocr_words_inside": segment.get("ocr_words_inside", ""),
            "text_preview": segment.get("text_preview", ""),
            "detection_method": segment.get("detection_method", ""),
            "confidence_note": segment.get("confidence_note", "")
        })

segmentation_summary_df = pd.DataFrame(segmentation_summary_rows)
segmented_regions_df = pd.DataFrame(segmented_region_rows)

segmentation_summary_csv_path = CSV_DIR / "segmentation_summary.csv"
segmented_regions_csv_path = CSV_DIR / "segmented_regions.csv"

segmentation_summary_df.to_csv(segmentation_summary_csv_path, index=False)
segmented_regions_df.to_csv(segmented_regions_csv_path, index=False)

segmentation_summary_df

In [ ]:
# inspect the segmentation annotation for the first document

example_segmentation_result = segmentation_results[0]
segmented_image_path = Path(example_segmentation_result["output_paths"]["annotated_segmentation_image"])

segmented_image = cv2.imread(str(segmented_image_path))

if segmented_image is None:
    raise FileNotFoundError(f"Could not load segmented image: {segmented_image_path}")

segmented_image_rgb = cv2.cvtColor(segmented_image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 14))
plt.imshow(segmented_image_rgb)
plt.axis("off")
plt.title(f"Phase 9 segmented regions: {example_segmentation_result['filename']}")
plt.show()

print("Segmented image:", segmented_image_path.name)
print("Segment count:", example_segmentation_result["segment_count"])

### Phase 9 notes

The segmentation is rule-based and combines OCR layout evidence with OpenCV feature detection.

This stage deliberately uses earlier outputs rather than introducing a new machine-learning model. That keeps the segmentation explainable for the presentation and matches the assignment requirement to demonstrate image segmentation into meaningful document regions.

The segmentation may still contain overlapping regions. That is acceptable at this stage because text sections, table candidates, and broad horizontal bands are different views of the same document structure.

# Multi-modal

## Stage 9 Multi-modal Plan

This section will later combine OCR, NLP, section analysis, visual feature detection, and segmented image regions into one document-level result.

The notebook now provides:

1. best OCR text per document
2. OCR confidence comparison
3. cleaned text, tokens, stems, and lemmas
4. TF-IDF keywords, key phrases, and regex-based entities
5. OCR line reconstruction and section categorisation
6. OpenCV visual feature detection
7. labelled image segmentation regions

The next phase will link OCR text and extracted NLP features directly to the segmented visual regions.

In [ ]:
# multi-modal development status for this commit

multimodal_development_status = {
    "ocr_text_extraction": "implemented",
    "ocr_confidence_comparison": "implemented",
    "ocr_word_csv_export": "implemented",
    "text_cleaning": "implemented",
    "tokenisation": "implemented",
    "stopword_removal": "implemented",
    "stemming": "implemented",
    "lemmatisation": "implemented",
    "tfidf_keyword_extraction": "implemented",
    "key_phrase_extraction": "implemented",
    "named_entity_extraction": "implemented",
    "document_reference_detection": "implemented",
    "ocr_line_reconstruction": "implemented",
    "section_categorisation": "implemented",
    "simple_table_row_extraction": "implemented",
    "basic_text_image_reference_detection": "implemented",
    "visual_feature_detection": "implemented",
    "horizontal_line_detection": "implemented",
    "table_like_region_detection": "implemented",
    "text_block_region_detection": "implemented",
    "signature_like_region_detection": "implemented",
    "diagram_or_figure_candidate_detection": "implemented",
    "annotated_visual_feature_images": "implemented",
    "image_segmentation": "implemented",
    "segmentation_json_export": "implemented",
    "segmentation_csv_export": "implemented",
    "annotated_segmentation_images": "implemented",
    "ocr_to_visual_region_mapping": "planned",
    "final_json_report_generation": "planned",
    "final_csv_summary_generation": "planned"
}

multimodal_development_status


# Final Output

In [ ]:
# final output summary for this development stage

# stage_2_summary = {
#     "stage": "document image loading",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "input_document_names": [document["filename"] for document in loaded_documents],
#     "next_stage": "add grayscale conversion, denoising, and threshold preprocessing"
# }
#print(json.dumps(stage_2_summary, indent=4))

# stage_3_summary = {
#     "stage": "OpenCV image preprocessing and enhancement",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "preprocessed_documents": len(preprocessed_documents),
#     "preprocessing_steps": [
#         "grayscale conversion",
#         "median blur denoising",
#         "CLAHE contrast enhancement",
#         "Otsu thresholding",
#         "adaptive thresholding",
#         "clean adaptive thresholding",
#         "Canny edge detection"
#     ],
#     "preprocessed_output_directory": str(PREPROCESSED_DIR),
#     "next_stage": "add Tesseract OCR extraction and OCR confidence reporting"
# }

# print(json.dumps(stage_3_summary, indent=4))

# stage_4_summary = {
#     "stage": "Tesseract OCR extraction and confidence comparison",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "preprocessed_documents": len(preprocessed_documents),
#     "ocr_documents_processed": len(ocr_results),
#     "ocr_candidate_sources": [
#         "gray",
#         "denoised",
#         "enhanced",
#         "adaptive_threshold_clean"
#     ],
#     "ocr_summary_csv": str(ocr_summary_csv_path),
#     "ocr_output_directory": str(OCR_DIR),
#     "ocr_text_output_directory": str(OCR_TEXT_DIR),
#     "next_stage": "add OCR text preprocessing with tokenisation, stopword removal, stemming, and lemmatisation"
# }

# print(json.dumps(stage_4_summary, indent=4))

# stage_5_summary = {
#     "stage": "OCR text preprocessing",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "preprocessed_documents": len(preprocessed_documents),
#     "ocr_documents_processed": len(ocr_results),
#     "nlp_documents_processed": len(nlp_results),
#     "nlp_preprocessing_steps": [
#         "OCR text normalisation",
#         "lowercase conversion",
#         "regex tokenisation",
#         "stopword removal",
#         "Porter stemming",
#         "WordNet lemmatisation"
#     ],
#     "nlp_output_directory": str(NLP_DIR),
#     "cleaned_text_output_directory": str(CLEANED_TEXT_DIR),
#     "text_preprocessing_summary_csv": str(nlp_summary_csv_path),
#     "next_stage": "add TF-IDF keyword extraction, key phrase extraction, and simple named entity extraction"
# }

# print(json.dumps(stage_5_summary, indent=4))

# stage_6_summary = {
#     "stage": "NLP feature recognition",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "preprocessed_documents": len(preprocessed_documents),
#     "ocr_documents_processed": len(ocr_results),
#     "nlp_documents_processed": len(nlp_results),
#     "text_feature_documents_processed": len(text_feature_results),
#     "feature_recognition_steps": [
#         "TF-IDF keyword extraction",
#         "simple key phrase extraction",
#         "regex-based date extraction",
#         "regex-based money value extraction",
#         "regex-based time extraction",
#         "regex-based URL extraction",
#         "regex-based phone number extraction",
#         "regex-based percentage extraction",
#         "regex-based document reference extraction",
#         "visual or structural reference term detection"
#     ],
#     "text_feature_output_directory": str(TEXT_FEATURE_DIR),
#     "text_feature_summary_csv": str(text_feature_summary_csv_path),
#     "next_stage": "add document text analysis and section categorisation"
# }

# print(json.dumps(stage_6_summary, indent=4))

# stage_7_summary = {
#     "stage": "text analysis and section categorisation",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "preprocessed_documents": len(preprocessed_documents),
#     "ocr_documents_processed": len(ocr_results),
#     "nlp_documents_processed": len(nlp_results),
#     "text_feature_documents_processed": len(text_feature_results),
#     "section_analysis_documents_processed": len(section_analysis_results),
#     "section_analysis_steps": [
#         "OCR line reconstruction from Tesseract word-level output",
#         "rule-based document section categorisation",
#         "simple table-like row extraction",
#         "basic text-image reference identification"
#     ],
#     "section_analysis_output_directory": str(SECTION_ANALYSIS_DIR),
#     "section_analysis_summary_csv": str(section_analysis_summary_csv_path),
#     "section_lines_csv": str(section_lines_csv_path),
#     "extracted_table_rows_csv": str(extracted_table_rows_csv_path),
#     "next_stage": "add OpenCV visual feature detection"
# }

# print(json.dumps(stage_7_summary, indent=4))

# stage_8_summary = {
#     "stage": "OpenCV visual feature detection",
#     "template_followed": True,
#     "input_documents_found": len(input_documents),
#     "successfully_loaded_documents": len(loaded_documents),
#     "preprocessed_documents": len(preprocessed_documents),
#     "ocr_documents_processed": len(ocr_results),
#     "nlp_documents_processed": len(nlp_results),
#     "text_feature_documents_processed": len(text_feature_results),
#     "section_analysis_documents_processed": len(section_analysis_results),
#     "visual_feature_documents_processed": len(visual_feature_results),
#     "visual_feature_detection_steps": [
#         "horizontal separator line detection",
#         "grid-like table region detection",
#         "table-like region detection between separator lines",
#         "text block region detection from OCR layout",
#         "signature-like connected component detection",
#         "diagram or figure candidate detection",
#         "annotated image generation"
#     ],
#     "visual_feature_output_directory": str(VISUAL_FEATURE_DIR),
#     "annotated_image_output_directory": str(ANNOTATED_DIR),
#     "visual_feature_summary_csv": str(visual_feature_summary_csv_path),
#     "visual_feature_regions_csv": str(visual_feature_regions_csv_path),
#     "next_stage": "add image segmentation into labelled document regions"
# }

# print(json.dumps(stage_8_summary, indent=4))


stage_9_summary = {
    "stage": "image segmentation into labelled regions",
    "template_followed": True,
    "input_documents_found": len(input_documents),
    "successfully_loaded_documents": len(loaded_documents),
    "preprocessed_documents": len(preprocessed_documents),
    "ocr_documents_processed": len(ocr_results),
    "nlp_documents_processed": len(nlp_results),
    "text_feature_documents_processed": len(text_feature_results),
    "section_analysis_documents_processed": len(section_analysis_results),
    "visual_feature_documents_processed": len(visual_feature_results),
    "segmentation_documents_processed": len(segmentation_results),
    "segmentation_steps": [
        "combined OCR line bounding boxes into section regions",
        "converted selected OpenCV visual features into segmentation regions",
        "created broad horizontal document bands from separator lines",
        "assigned segment IDs and labels",
        "saved annotated segmentation images",
        "exported segmentation JSON and CSV outputs"
    ],
    "segmentation_output_directory": str(SEGMENTATION_DIR),
    "segmentation_summary_csv": str(segmentation_summary_csv_path),
    "segmented_regions_csv": str(segmented_regions_csv_path),
    "next_stage": "link OCR text and extracted features to segmented visual regions"
}

print(json.dumps(stage_9_summary, indent=4))

In [ ]:
# development notes
# commit 9 adds image segmentation into labelled document regions
# segmentation combines OCR section boxes, OpenCV visual features, and horizontal separator bands
# segmentation records are saved to outputs/segmentation and outputs/csv_reports
# annotated segmentation images are saved to outputs/annotated_images
# text-image integration is intentionally left for the next phase